# Assignment 3: Introduction to Spark Structured Streaming

## Part A: Theory

### What is Structured Streaming in Spark?

Structured Streaming is a scalable stream processing engine built on Spark SQL. It treats live data streams as unbounded tables that continuously grow with new data.

### How it Differs from Batch Processing

| Feature | Batch Processing | Structured Streaming |
|---------|-----------------|---------------------|
| Data Type | Fixed, static datasets | Continuous, unbounded streams |
| Processing | One-time execution | Continuous processing |
| Latency | Minutes to hours | Seconds to milliseconds |
| API | spark.read | spark.readStream |

### Key Features

1. Unified API for batch and streaming
2. Fault tolerance with exactly-once guarantees
3. Event-time processing with watermarks
4. Multiple output modes (Append, Complete, Update)
5. Scalable across clusters

### Real-Time Use Cases

1. Fraud detection in financial transactions
2. Real-time recommendation systems
3. IoT sensor monitoring
4. Social media sentiment analysis
5. Network monitoring and alerts

## Part B: Practical Implementation

In [0]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col, array, lit, element_at, when
import time

In [0]:
# Create SparkSession (Databricks provides this automatically)
spark = SparkSession.builder.appName("StructuredStreamingDemo").getOrCreate()
print("SparkSession ready")

SparkSession ready


In [0]:
# Define sample sentences for simulation
sentences = [
    "Apache Spark is a unified analytics engine",
    "Spark Structured Streaming makes it easy to build streaming applications",
    "Streaming data processing is important for real-time analytics",
    "Structured Streaming provides exactly-once semantics",
    "Apache Spark handles fault tolerance automatically"
]

print(f"Prepared {len(sentences)} sample sentences")

Prepared 5 sample sentences


In [0]:
# Create streaming source using rate (generates rows automatically)
streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

# Map generated numbers to sentences
text_stream = streaming_df.withColumn(
    "line",
    element_at(
        array(*[lit(s) for s in sentences]),
        (col("value") % len(sentences) + 1).cast("int")
    )
)

print("Streaming DataFrame created")
text_stream.printSchema()

Streaming DataFrame created
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- line: string (nullable = false)



In [0]:
# Split lines into words
words_df = text_stream.select(explode(split(col("line"), " ")).alias("word"))
print("Words extracted from lines")

Words extracted from lines


In [0]:
# Count occurrences of each word
word_counts = words_df.groupBy("word").count()
print("Word count aggregation ready")

Word count aggregation ready


In [0]:
# Display streaming results (Databricks display)
display(word_counts)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5809641707447870>, line 2
      1 # Display streaming results (Databricks display)
----> 2 display(word_counts)

File /databricks/python_shell/lib/dbruntime/display.py:142, in Display.display(self, input, *args, **kwargs)
    140 # This version is for Serverless + Spark Connect dogfooding.
    141 elif self.spark_connect_enabled and isinstance(input, ConnectDataFrame):
--> 142     self.display_connect_table(input, **kwargs)
    143 elif isinstance(input, ConnectDataFrame):
    144     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:104, in Display.display_connect_table(self, df, **kwargs)
     99     raise type(
    100         e
    101     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
    102       ) from e
    103 if df

In [0]:
# Stop all streaming queries
for query in spark.streams.active:
    query.stop()
    print(f"Stopped query: {query.id}")

print("All streams stopped")

## Bonus: Even/Odd Classification

In [0]:
# Create rate stream
rate_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

# Classify numbers as even or odd
classified_df = rate_df.withColumn(
    "parity",
    when(col("value") % 2 == 0, "even").otherwise("odd")
)

# Count even and odd numbers
parity_counts = classified_df.groupBy("parity").count()

# Display results
display(parity_counts)

In [0]:
# Stop all streaming queries
for query in spark.streams.active:
    query.stop()
    print(f"Stopped query: {query.id}")

print("Demo complete")

## Summary

### Steps Completed:

1. Created SparkSession
2. Generated streaming data using rate source
3. Mapped data to sample sentences
4. Split sentences into words
5. Counted word frequencies
6. Displayed streaming results
7. Demonstrated even/odd classification
8. Cleaned up all streaming queries

### Key Concepts:

- readStream: Creates streaming DataFrame
- rate source: Generates test data
- split and explode: Text processing
- groupBy and count: Aggregation
- display: Shows streaming output
- Output modes: Complete, Append, Update